In [3]:
import pandas as pd
import numpy as np
import os
import timeit

rS = timeit.default_timer()

if str(os.getlogin()) == 'chakis':
    folder_path = \
        r'C:\\Users\\chakis\\Downloads\\sandbox mode\\obc\\xls raw\\mpr raw\\'
elif str(os.getlogin()) == 'tiftor':
    folder_path = \
        r'C:\\Users\\tiftor\\Downloads\\'
elif str(os.getlogin()) == 'marund':
    folder_path = \
        r'C:\\Users\\marund\\Desktop\\Python'
elif str(os.getlogin()) == 'swasaw':
    folder_path = \
        r'C:\\Users\\swasaw\\Downloads\\'

    
#--Change the date based on reporting week
dS = '2025-04-07'
csv_name = dS + ' ' + '-' + ' ' + 'MPR with only approved and ready relievers'

df = pd.read_csv(folder_path + csv_name + '.csv', skiprows=3,
                 error_bad_lines=False)
df.columns = df.columns.map(str.upper)

df.rename(columns={
    'SEAFARER_MOB_CELL': 'MOBILISATION_CELL',
    'SEAFARER_PLANNING_CELL': 'PLANNING_CELL',
    'SUPPLYTYPE': 'SUPPLY_TYPE',
    'CREW_SUPPLY_OFFICE': 'SFC',
    'VES_IMONUMBER': 'IMO_NUMBER',
    'SIGNONDATE': 'SIGNON_DATE',
    'SIGNONPORT': 'SIGNON_PORT',
    'DUERELEIF': 'DUE_RELEIF',
    'RELIEVERNOTREQUIRED': 'RELIEVER_NOT_REQUIRED',
    'CONTRACTDAYS': 'CONTRACT_DAYS',
    'EXTENSIONDAYS': 'EXTENSION_DAYS',
    'EXTENSION_AGREEDBYSEAFARER': 'EXTENSION_AGREED',
    'BUDGETBERTHDAYS': 'BUDGET_BERTH_DAYS',
    'MNGMT_START_DATE': 'MANAGEMENT_START',
    'LASTUPDATEDBY': 'UPDATED_BY',
    'LASTUPDATED': 'UPDATED_ON',
    }, inplace=True)

df[['WB_MON', 'DATE_EXTRACTED']] = pd.to_datetime(dS)

df[['SIGNON_DATE', 'DUE_RELEIF', 'MANAGEMENT_START', 'LASTAPPROVEDDATE'
   ]] = df[['SIGNON_DATE', 'DUE_RELEIF', 'MANAGEMENT_START',
           'LASTAPPROVEDDATE']].apply(pd.to_datetime)

df['WB_MON'] = df['WB_MON'].dt.date
df['DATE_EXTRACTED'] = df['DATE_EXTRACTED'].dt.date
df['LASTAPPROVEDDATE'] = df['LASTAPPROVEDDATE'].dt.date
df['SIGNON_DATE'] = df['SIGNON_DATE'].dt.date
df['DUE_RELEIF'] = df['DUE_RELEIF'].dt.date
df['MANAGEMENT_START'] = df['MANAGEMENT_START'].dt.date

# df.info()

# '-------------------cleaning section'
#remove logic of changing IMO #s this should be captured as it is to avoid look up discrepancies

#change = ' .*/\[]$%+!@#)(&^-=_,'
#for char in change:
#    df['IMO_NUMBER'] = df['IMO_NUMBER'].str.replace(char, '',
#            regex=True)

#x = 0
#while x < 10:
#    df['IMO_NUMBER'] = df['IMO_NUMBER'].str.replace(str(x) + 'A',
#            str(x), regex=True)
#    df['IMO_NUMBER'] = df['IMO_NUMBER'].str.replace(str(x) + 'B',
#            str(x), regex=True)
#    df['IMO_NUMBER'] = df['IMO_NUMBER'].str.replace(str(x) + 'w',
#            str(x), regex=True)
#    df['IMO_NUMBER'] = df['IMO_NUMBER'].str.replace(str(x) + 'x',
#            str(x), regex=True)
#    df['IMO_NUMBER'] = df['IMO_NUMBER'].str.replace(str(x) + 'y',
#            str(x), regex=True)
#    x = x + 1

#df['IMO_NUMBER'] = df['IMO_NUMBER'].fillna(df['VESSEL_NAME'])
#df['PCN'] = df['PCN'].fillna(df['SEAFARER_SURNAME'])

repz = 0
v1 = 'x'
v2 = 'x'
while repz < 8:
    if repz == 1:
        v1 = 'Crew Mgmt'
        v2 = 'CREW MANAGEMENT'
    elif repz == 2:
        v1 = 'Tech Mgmt'
        v2 = 'SHIP MANAGEMENT'
    elif repz == 3:
        v1 = 'TECH Mgmt'
        v2 = 'SHIP MANAGEMENT'
    elif repz == 4:
        v1 = 'FULL Management'
        v2 = 'SHIP MANAGEMENT'
    elif repz == 5:
        v1 = 'Full Management'
        v2 = 'SHIP MANAGEMENT'
    elif repz == 6:
        v1 = 'Full Management'
        v2 = 'SHIP MANAGEMENT'
    elif repz == 7:
        v1 = 'TP Crew Mgmt'
        v2 = 'TP CREW MANAGEMENT'

    df = df.replace(str(v1), str(v2))

    repz = repz + 1

repz = 0

# v1 = 'International Tanker Management Limited., Dubai Branch'
# v2 = 'ITM DUBAI'

while repz < 4:
    if repz == 1:
        v1 = 'International Tanker Management Limited., Dubai Branch'
        v2 = 'ITM DUBAI'
        
        
#         ###commented out split for ASIA AND ABERDEEN
#     if repz == 2:
#         v1 = 'V.Ships Offshore Limited (Aberdeen)'
#         v2 = 'V.SHIPS OFFSHORE (ASIA & ABERDEEN)'
#     if repz == 3:
#         v1 = 'V.Ships Offshore (Asia) Pte. Ltd'
#         v2 = 'V.SHIPS OFFSHORE (ASIA & ABERDEEN)'

    df = df.replace(str(v1), str(v2))

    repz = repz + 1

df['UPDATED_ON'] = pd.to_datetime(df['UPDATED_ON'])
df.sort_index(axis=0, ascending=False, inplace=True)

# do not delete vessels left as long as it has positions onboard--

#df.drop(df.index[df['VESSEL_STATUS'] == 'Left Management'],
#        inplace=True)
#df.drop(df.index[df['VESSEL_STATUS'] == 'Left Management run out period'
#        ], inplace=True)

df.drop(df.index[df['VESSEL_NAME'] == 'TESTSHIP'], inplace=True)

df['rs_index'] = df['RELIEVER_STATUS'].map({
    'Proposed': 'a',
    'Plan Proposed': 'b',
    'Planned': 'c',
    'Approved': 'd',
    'Ready': 'e',
    })
df['rs_index'] = df['rs_index'].fillna('f')

df['temp_a'] = df.sort_values(['UPDATED_ON'],
                              ascending=False).groupby(['PCN'
        ]).cumcount() + 1

df.drop(df.index[df['temp_a'] > 1], inplace=True)
df.drop(['temp_a', 'rs_index'], axis=1, inplace=True)

# df['VESSEL_MANAGEMENT_TYPE'].unique()
# '-------------------save section'

csv_name = 'mpr relievers cleaned'

df.to_excel(str(folder_path) + str(dS) + ' ' + '-' + ' ' + str(csv_name) + '.xlsx',
            index=False, sheet_name='data', header=True)

rE = timeit.default_timer()
print ('Time: ', rE - rS)

C:\Users\tiftor\AppData\Local\Temp\ipykernel_24812\3416577691.py:26: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  df = pd.read_csv(folder_path + csv_name + '.csv', skiprows=3,


Time:  15.315419699999893
